# Star catalogue for F-CAMs

In preparation of the scvPIC, we need prepare a list of 300 stars for observations with PLATO F-CAMs. The main idea behind this sample is to thoroughly test and improve theory of stellar structure and evolution across the HRD. The sample needs to be prepared by the upcoming Monday, and we need your help on this front. To begin with, could you please select stars with $P \leq 10$ from your Gaia magnitude-limited catalog in LOPS2. Then, from the resulting sub-sample, could you please select the following categories using classifications as listed in Simbad:

- Blue supergiants
- Asymptotic Giant Branch (AGB) stars
- Post Asymptotic Giant Branch (post-AGB) stars
- Contact/interacting binaries, spectral types O and B
- Algol-type binaries

It would be fabulous if you could deliver the lists by the end of this week. It will suffice if these lists contain the following columns: 1) Gaia DR3 ID, 2) RA & DEC, 3) G-band magnitude

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib notebook

In [ ]:
import os
import sys
import glob
import scipy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm 

# PlatoSim libraries
import platosim.plot      as pt
import platosim.starquery as sq
import platosim.utilities as ut
from platosim.matplotlibrc import setup_paper
setup_paper(warning=False)

from IPython.display import display, HTML
display(HTML("<style>.container {width:80% !important;}</style>"))

In [ ]:
# Define paths used throughout
path = Path(os.getenv('PLATO_WORKDIR')) / 'PlatoCS'
fdir = path / 'plots'

# Use seed for reproducebility
seed = 12345
rng = ut.rng(seed)

## Load PLATO-CS F-CAM catalogue

In [ ]:
# Load F-CAM catalogue
df = pd.read_feather(path / 'starcat_Gmag17' / 'starcat_GaiaDR3_FCAM_LOPS2.ftr')

In [ ]:
# Fetch bright stars
dt = df[df.Pmag <= 10].reset_index(drop=True)
dt

In [ ]:
# Fetch OTYPE of Simbad to assemble catalogue
for i in tqdm(range(dt.shape[0]), bar_format=ut.tqdmBar()):
    di = sq.simbadQuery(f'Gaia DR3 {dt.gaiaDR3.iloc[i]}')
    if i == 0: dq = di
    else: dq = pd.concat([dq, di])

In [ ]:
# Correct data frame
dq = dq.reset_index(drop=True)
dt['otype'] = dq.otype
dt.insert(0, 'source', dq.source)
dt = dt.drop(columns='ncams')

The following OTYPE names are used:

- Blue supergiants: `BlueSG`
- Asymptotic Giant Branch (AGB) stars: `AGB*`
- Post Asymptotic Giant Branch (post-AGB) stars: `post-AGB*`
- Contact/interacting binaries, spectral types O and B: ``
- Algol-type binaries: `Al*`

Reference: `https://simbad.cds.unistra.fr/Pages/guide/otypes.htx`

In [ ]:
# Save catalogue
dt.to_feather(path / 'starcat_Gmag17' / 'starcat_GaiaDR3_FCAM_LOPS2_Pmax10.ftr')

In [ ]:
dq[dq.otype == 'BlueSG']

In [ ]:
dq[dq.otype == 'AGB*']

In [ ]:
dq[dq.otype == 'post-AGB*']

In [ ]:
dq[dq.otype == '**']

In [ ]:
dt_BlueSG  = dt[dt.otype == 'BlueSG']
dt_AGB     = dt[dt.otype == 'AGB*']
dt_postAGB = dt[dt.otype == 'post-AGB*']
dt_binary  = dt[dt.otype == '**']

### Cross-match with EBs (IJspeert+2024)

In [ ]:
# Download file from FTP
ut.downloadFromFTP('table_OBAF_EB_catalogue.csv')

In [ ]:
# Load catalogue
idir = Path(os.getenv('PLATO_PROJECT_HOME')) / 'inputfiles'
db = pd.read_csv(idir / 'table_OBAF_EB_catalogue.csv')

In [ ]:
# Cross-match with Luc's catalogue
dt.gaiaDR3 = dt.gaiaDR3.astype(float)
db0 = db[db['GAIA_DR3'].isin(dt['gaiaDR3'])].sort_values(by=['GAIA_DR3']).reset_index(drop=True)
dt0 = dt[dt['gaiaDR3'].isin(db['GAIA_DR3'])].sort_values(by=['gaiaDR3']).reset_index(drop=True)

In [ ]:
# Select B stars (no O stars are available)
dt_EBs = dt0[dt0.spec == 'B']
dt_EBs

### Plot star in sky

In [ ]:
# Show pointign field
fig, ax = pt.plotPlatoFOV('LOPS2', raStars=dt.ra, decStars=dt.dec,
                          showGroups=False, showFcamFOV=False, 
                          fovSize=20, fs=20, ms=5, figsize=(8,8))

# Plot requested variable stars
zorder = 10
ax.scatter(dt_double.ra, dt_double.dec, s=20, marker='o', edgecolor='w', facecolor='green',
           linewidth=0.5, transform=ax.get_transform('world'), zorder=zorder)
ax.scatter(dt_EBs.ra, dt_EBs.dec, s=20, marker='o', edgecolor='w', facecolor='deeppink', 
           linewidth=0.5, transform=ax.get_transform('world'), zorder=zorder)
ax.scatter(dt_BlueSG.ra, dt_BlueSG.dec, s=40, marker='o', edgecolor='w', facecolor='blue', 
           linewidth=0.5, transform=ax.get_transform('world'), zorder=zorder)
# ax.scatter(dt_AGB.ra, dt_AGB.dec, s=40, marker='o', edgecolor='w', facecolor='tomato', 
#            linewidth=0.5, transform=ax.get_transform('world'), zorder=zorder)
ax.scatter(dt_postAGB.ra, dt_postAGB.dec, s=40, marker='o', edgecolor='w', facecolor='orange', 
           linewidth=0.5, transform=ax.get_transform('world'), zorder=zorder)

fig.savefig(fdir / 'starcat_GaiaDR3_FCAM_LOPS2.png', bbox_inches='tight', dpi=300);